In [ ]:
#install dependencies
import subprocess, sys
PKGS = ['pandas==2.2.2','numpy==1.26.4','scikit-learn==1.5.0',
        'xgboost==2.0.3','lightgbm','shap==0.45.1']
for p in PKGS:
    subprocess.check_call([sys.executable,'-m','pip','install',p,'-q',
                           '--break-system-packages'])
print('Install selesai')

Install selesai


In [2]:
#impor library
import numpy as np, pandas as pd, pickle, os, warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, top_k_accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap
np.random.seed(42)
print('Import selesai')

Import selesai


In [ ]:
#jurusan beserta profil ideal
#[mtk,fis,kim,bio,bind,bing,eko,sos,sej,seni, R,I,A,S,E,C, peminatan]
#peminatan: 0=IPA, 1=IPS, 2=Bahasa

JURUSAN_PROFILES = {
    # ─ Teknik & Informatika ────────────────────────────────
    'Teknik Informatika':         [93,76,62,52, 66,82, 52,44,42,42,  3,9,5,3,6,7, 0],
    'Ilmu Komputer':              [95,78,62,52, 67,84, 52,44,42,42,  3,9,5,3,5,7, 0],
    'Sistem Informasi':           [85,62,52,48, 70,80, 78,56,48,52,  3,7,5,5,8,8, 0],
    'Rekayasa Perangkat Lunak':   [90,72,58,48, 66,82, 58,44,42,44,  4,8,6,3,6,7, 0],
    'Data Science':               [95,75,64,58, 68,86, 80,56,48,42,  2,9,4,4,6,7, 0],
    'Kecerdasan Buatan':          [94,76,62,54, 67,85, 65,48,44,44,  3,9,5,3,6,6, 0],
    'Cyber Security':             [90,78,60,50, 66,82, 55,44,44,42,  4,9,4,2,5,7, 0],
    'Teknik Elektro':             [88,94,68,52, 62,76, 52,38,40,40,  7,8,4,2,5,6, 0],
    'Teknik Mesin':               [86,92,66,52, 60,74, 52,38,40,42,  9,7,3,2,5,5, 0],
    'Teknik Sipil':               [90,88,64,50, 68,78, 58,44,40,50,  8,7,4,3,5,6, 0],
    'Teknik Kimia':               [86,82,95,64, 63,76, 52,40,40,42,  6,8,4,2,5,5, 0],
    'Teknik Industri':            [86,76,60,50, 70,78, 82,56,48,50,  6,7,4,4,8,7, 0],
    'Teknik Perminyakan':         [88,90,84,64, 62,74, 72,42,42,40,  8,8,3,2,7,6, 0],
    'Teknik Lingkungan':          [80,74,80,88, 68,76, 56,52,46,48,  4,7,4,6,5,5, 0],
    'Arsitektur':                 [82,74,58,50, 70,78, 50,54,50,93,  4,6,9,4,6,5, 0],
    'Perencanaan Wilayah & Kota': [78,68,54,52, 74,78, 74,70,60,60,  4,6,7,5,7,5, 0],
    # ─ Sains & MIPA ────────────────────────────────────────
    'Matematika':                 [97,86,70,60, 64,74, 52,40,40,40,  2,9,4,3,4,6, 0],
    'Statistika':                 [94,78,64,56, 66,76, 72,48,42,40,  2,9,4,4,5,7, 0],
    'Aktuaria':                   [96,74,60,50, 68,78, 93,46,42,40,  2,9,3,3,7,8, 0],
    'Fisika':                     [88,96,74,60, 64,74, 48,38,40,40,  3,9,3,2,4,5, 0],
    'Kimia':                      [84,76,97,72, 64,74, 48,40,40,40,  3,8,4,4,4,5, 0],
    'Biologi':                    [76,68,78,96, 68,76, 52,56,42,46,  3,8,5,6,4,5, 0],
    'Bioteknologi':               [82,72,86,94, 68,78, 52,50,42,44,  4,8,5,5,4,5, 0],
    'Astronomi':                  [92,97,72,58, 63,78, 46,38,44,42,  2,9,3,2,3,5, 0],
    'Geofisika':                  [90,94,72,60, 63,76, 48,38,44,40,  4,8,3,2,4,5, 0],
    'Oseanografi':                [80,80,74,84, 66,74, 50,44,46,44,  5,8,4,4,4,5, 0],
    # ─ Kesehatan & Kedokteran ──────────────────────────────
    'Kedokteran':                 [84,78,90,95, 74,82, 52,64,46,46,  4,8,3,8,5,4, 0],
    'Kedokteran Gigi':            [78,70,86,90, 70,78, 50,62,42,62,  4,7,3,7,5,5, 0],
    'Farmasi':                    [82,72,95,90, 70,80, 52,56,40,46,  3,8,4,6,5,6, 0],
    'Keperawatan':                [70,60,72,88, 78,76, 48,82,48,48,  2,6,3,9,4,4, 0],
    'Kebidanan':                  [66,54,70,90, 76,72, 46,84,44,50,  2,5,3,9,4,4, 0],
    'Kesehatan Masyarakat':       [74,58,68,82, 80,78, 72,78,58,48,  2,7,3,8,6,5, 0],
    'Gizi':                       [73,56,78,86, 72,76, 52,66,40,50,  3,7,4,7,4,5, 0],
    'Kedokteran Hewan':           [76,68,76,94, 68,74, 50,64,42,44,  4,7,3,8,4,4, 0],
    'Fisioterapi':                [70,66,68,86, 76,74, 48,82,44,52,  3,6,3,9,4,4, 0],
    'Analis Kesehatan':           [78,64,84,86, 68,74, 48,60,40,44,  3,7,3,6,4,6, 0],
    # ─ Ekonomi & Bisnis ────────────────────────────────────
    'Akuntansi':                  [84,52,42,40, 74,76, 95,56,52,50,  2,5,3,5,7,9, 1],
    'Manajemen':                  [74,48,38,36, 78,80, 93,66,58,56,  2,4,4,6,9,7, 1],
    'Ilmu Ekonomi':               [86,50,38,36, 76,80, 95,60,58,50,  2,7,3,5,8,7, 1],
    'Keuangan':                   [85,54,44,38, 74,80, 95,58,52,48,  2,6,3,4,8,8, 1],
    'Perbankan & Keuangan':       [83,52,42,38, 74,78, 93,60,52,48,  2,5,3,4,8,8, 1],
    'Manajemen Pemasaran':        [70,44,36,34, 80,84, 90,70,52,62,  2,4,5,6,9,6, 1],
    'Bisnis Digital':             [82,58,46,42, 76,86, 88,62,52,62,  3,6,6,5,9,7, 1],
    'Manajemen SDM':              [66,42,34,34, 80,80, 84,82,58,56,  2,3,4,8,8,5, 1],
    # ─ Hukum & Ilmu Sosial ─────────────────────────────────
    'Hukum':                      [64,40,34,34, 93,84, 64,74,76,46,  1,5,4,7,8,5, 1],
    'Hukum Islam (Syariah)':      [60,36,32,32, 93,76, 60,78,78,48,  1,5,4,8,7,5, 1],
    'Ilmu Politik':               [60,38,34,34, 86,82, 62,88,86,48,  1,5,4,8,7,4, 1],
    'Hubungan Internasional':     [60,36,34,32, 88,95, 62,76,74,52,  1,5,5,7,8,4, 1],
    'Sosiologi':                  [58,36,34,36, 84,78, 56,94,76,56,  1,5,5,9,5,3, 1],
    'Antropologi':                [54,34,30,32, 84,78, 50,93,88,58,  1,5,5,9,5,3, 1],
    'Psikologi':                  [68,44,40,40, 82,82, 60,93,72,62,  2,6,5,9,6,4, 1],
    'Administrasi Publik':        [66,44,36,36, 82,80, 76,80,72,50,  2,5,4,7,8,6, 1],
    'Ilmu Pemerintahan':          [60,38,34,34, 84,80, 66,85,82,48,  1,5,4,8,7,5, 1],
    'Kriminologi':                [60,38,34,34, 86,80, 58,88,80,46,  2,6,4,8,6,4, 1],
    # ─ Pendidikan & Keguruan ───────────────────────────────
    'Pendidikan Guru SD':         [72,52,48,50, 86,82, 60,82,68,62,  2,5,5,9,5,5, 1],
    'Pendidikan Anak Usia Dini':  [60,42,38,46, 84,78, 54,92,60,74,  1,3,7,9,4,3, 1],
    'Pendidikan Matematika':      [93,74,60,52, 80,78, 50,72,52,54,  2,7,4,8,5,5, 1],
    'Pendidikan Fisika':          [85,92,66,54, 74,76, 46,70,48,48,  3,8,4,8,5,5, 0],
    'Pendidikan Kimia':           [80,70,90,68, 74,76, 46,70,46,48,  3,7,4,8,5,5, 0],
    'Pendidikan Biologi':         [72,64,72,90, 74,76, 48,72,46,50,  3,7,4,8,4,5, 0],
    'Pendidikan Bahasa Indonesia':[60,38,34,34, 97,82, 50,74,72,68,  1,3,6,9,4,4, 2],
    'Pendidikan Bahasa Inggris':  [58,36,32,32, 86,97, 50,72,68,66,  1,3,6,8,5,4, 2],
    'Pendidikan Sejarah':         [52,34,30,32, 84,78, 52,76,96,52,  1,4,4,8,5,4, 1],
    'Pendidikan Ekonomi':         [76,46,36,34, 80,76, 90,72,60,52,  2,5,4,7,7,6, 1],
    'Pendidikan Seni Rupa':       [50,34,30,32, 74,72, 42,64,48,97,  1,2,9,7,4,3, 2],
    'Bimbingan Konseling':        [56,36,30,32, 84,80, 52,96,62,56,  1,4,5,9,5,3, 1],
    # ─ Komunikasi & Media ──────────────────────────────────
    'Ilmu Komunikasi':            [58,36,32,32, 93,86, 64,86,64,78,  1,4,7,8,8,4, 1],
    'Jurnalistik':                [54,34,30,30, 96,86, 58,82,76,72,  1,4,7,8,7,3, 1],
    'Penyiaran (Broadcasting)':   [52,34,30,30, 90,86, 60,80,64,86,  1,3,8,7,7,3, 2],
    'Public Relations':           [54,34,30,30, 90,86, 72,86,58,68,  1,3,6,8,8,4, 1],
    'Periklanan':                 [58,38,32,30, 82,84, 74,76,52,88,  2,4,8,6,8,4, 2],
    'Pariwisata':                 [56,36,32,32, 85,92, 72,82,60,74,  1,3,7,8,8,4, 1],
    'Perhotelan':                 [52,34,30,32, 82,90, 70,84,54,72,  1,2,6,9,7,4, 1],
    # ─ Seni & Desain ───────────────────────────────────────
    'Desain Komunikasi Visual':   [64,50,42,38, 74,82, 52,60,48,97,  3,5,10,5,6,4, 2],
    'Desain Grafis':              [62,50,42,38, 72,78, 50,60,48,94,  3,5,9,5,6,4, 2],
    'Animasi':                    [70,56,46,40, 70,80, 50,54,46,96,  3,5,10,4,5,4, 2],
    'Seni Rupa':                  [48,32,30,32, 72,70, 42,60,48,99,  1,2,10,6,3,2, 2],
    'Seni Musik':                 [46,28,28,28, 68,76, 42,58,46,98,  1,2,10,5,4,2, 2],
    'Film dan Televisi':          [56,36,32,32, 82,84, 50,72,58,94,  1,4,9,7,6,3, 2],
    'Fotografi':                  [54,38,32,32, 76,80, 48,62,48,96,  2,4,10,5,5,3, 2],
    'Seni Tari':                  [40,28,26,28, 70,72, 40,66,50,98,  1,1,10,7,3,1, 2],
    # ─ Bahasa & Sastra ─────────────────────────────────────
    'Sastra Indonesia':           [52,32,28,28, 98,78, 48,68,72,74,  1,3,7,8,5,3, 2],
    'Sastra Inggris':             [52,32,28,28, 88,98, 48,64,64,84,  1,3,8,7,5,3, 2],
    'Sastra Jepang':              [52,32,28,28, 80,93, 48,62,58,78,  1,2,8,7,5,3, 2],
    'Sastra Arab':                [50,30,28,30, 90,84, 46,66,62,70,  1,2,7,8,5,3, 2],
    'Sastra Mandarin':            [54,34,28,28, 76,90, 52,62,56,76,  1,2,8,7,5,3, 2],
    'Linguistik':                 [66,40,34,34, 94,94, 56,68,66,62,  1,5,6,7,5,4, 2],
    'Ilmu Perpustakaan':          [62,40,34,34, 86,80, 62,68,70,54,  1,5,4,7,5,6, 1],
    # ─ Pertanian & Lingkungan ──────────────────────────────
    'Agroteknologi':              [76,66,72,88, 66,72, 58,60,46,44,  5,7,4,6,5,5, 0],
    'Agribisnis':                 [70,54,50,68, 72,76, 90,70,58,46,  3,5,4,6,8,7, 1],
    'Peternakan':                 [68,58,66,86, 64,68, 58,64,44,42,  5,6,3,7,5,5, 0],
    'Perikanan':                  [72,66,68,84, 64,70, 58,62,46,44,  5,7,3,6,5,5, 0],
    'Kehutanan':                  [72,66,70,88, 66,72, 54,62,52,48,  5,7,4,6,4,5, 0],
    'Teknologi Pangan':           [76,64,84,82, 68,74, 64,54,42,46,  4,7,4,5,5,6, 0],
    'Ilmu Tanah':                 [74,64,74,84, 63,70, 54,52,46,40,  5,7,4,5,4,5, 0],
}

# ── Mapping jurusan → rumpun ──────────────────────────────────
RUMPUN_MAP = {
    'Teknik Informatika':'Teknik & Informatika',   'Ilmu Komputer':'Teknik & Informatika',
    'Sistem Informasi':'Teknik & Informatika',     'Rekayasa Perangkat Lunak':'Teknik & Informatika',
    'Data Science':'Teknik & Informatika',         'Kecerdasan Buatan':'Teknik & Informatika',
    'Cyber Security':'Teknik & Informatika',       'Teknik Elektro':'Teknik & Informatika',
    'Teknik Mesin':'Teknik & Informatika',         'Teknik Sipil':'Teknik & Informatika',
    'Teknik Kimia':'Teknik & Informatika',         'Teknik Industri':'Teknik & Informatika',
    'Teknik Perminyakan':'Teknik & Informatika',   'Teknik Lingkungan':'Teknik & Informatika',
    'Arsitektur':'Teknik & Informatika',           'Perencanaan Wilayah & Kota':'Teknik & Informatika',
 
    'Matematika':'Sains & MIPA',   'Statistika':'Sains & MIPA',    'Aktuaria':'Sains & MIPA',
    'Fisika':'Sains & MIPA',       'Kimia':'Sains & MIPA',          'Biologi':'Sains & MIPA',
    'Bioteknologi':'Sains & MIPA', 'Astronomi':'Sains & MIPA',      'Geofisika':'Sains & MIPA',
    'Oseanografi':'Sains & MIPA',
 
    'Kedokteran':'Kesehatan & Kedokteran',         'Kedokteran Gigi':'Kesehatan & Kedokteran',
    'Farmasi':'Kesehatan & Kedokteran',            'Keperawatan':'Kesehatan & Kedokteran',
    'Kebidanan':'Kesehatan & Kedokteran',          'Kesehatan Masyarakat':'Kesehatan & Kedokteran',
    'Gizi':'Kesehatan & Kedokteran',               'Kedokteran Hewan':'Kesehatan & Kedokteran',
    'Fisioterapi':'Kesehatan & Kedokteran',        'Analis Kesehatan':'Kesehatan & Kedokteran',
 
    'Akuntansi':'Ekonomi & Bisnis',       'Manajemen':'Ekonomi & Bisnis',
    'Ilmu Ekonomi':'Ekonomi & Bisnis',    'Keuangan':'Ekonomi & Bisnis',
    'Perbankan & Keuangan':'Ekonomi & Bisnis', 'Manajemen Pemasaran':'Ekonomi & Bisnis',
    'Bisnis Digital':'Ekonomi & Bisnis',  'Manajemen SDM':'Ekonomi & Bisnis',
 
    'Hukum':'Hukum & Ilmu Sosial',                'Hukum Islam (Syariah)':'Hukum & Ilmu Sosial',
    'Ilmu Politik':'Hukum & Ilmu Sosial',          'Hubungan Internasional':'Hukum & Ilmu Sosial',
    'Sosiologi':'Hukum & Ilmu Sosial',             'Antropologi':'Hukum & Ilmu Sosial',
    'Psikologi':'Hukum & Ilmu Sosial',             'Administrasi Publik':'Hukum & Ilmu Sosial',
    'Ilmu Pemerintahan':'Hukum & Ilmu Sosial',     'Kriminologi':'Hukum & Ilmu Sosial',
 
    'Pendidikan Guru SD':'Pendidikan & Keguruan',
    'Pendidikan Anak Usia Dini':'Pendidikan & Keguruan',
    'Pendidikan Matematika':'Pendidikan & Keguruan',
    'Pendidikan Fisika':'Pendidikan & Keguruan',   'Pendidikan Kimia':'Pendidikan & Keguruan',
    'Pendidikan Biologi':'Pendidikan & Keguruan',
    'Pendidikan Bahasa Indonesia':'Pendidikan & Keguruan',
    'Pendidikan Bahasa Inggris':'Pendidikan & Keguruan',
    'Pendidikan Sejarah':'Pendidikan & Keguruan',  'Pendidikan Ekonomi':'Pendidikan & Keguruan',
    'Pendidikan Seni Rupa':'Pendidikan & Keguruan','Bimbingan Konseling':'Pendidikan & Keguruan',
 
    'Ilmu Komunikasi':'Komunikasi & Media',   'Jurnalistik':'Komunikasi & Media',
    'Penyiaran (Broadcasting)':'Komunikasi & Media', 'Public Relations':'Komunikasi & Media',
    'Periklanan':'Komunikasi & Media',        'Pariwisata':'Komunikasi & Media',
    'Perhotelan':'Komunikasi & Media',
 
    'Desain Komunikasi Visual':'Seni & Desain', 'Desain Grafis':'Seni & Desain',
    'Animasi':'Seni & Desain',    'Seni Rupa':'Seni & Desain',
    'Seni Musik':'Seni & Desain', 'Film dan Televisi':'Seni & Desain',
    'Fotografi':'Seni & Desain',  'Seni Tari':'Seni & Desain',
 
    'Sastra Indonesia':'Bahasa & Sastra',  'Sastra Inggris':'Bahasa & Sastra',
    'Sastra Jepang':'Bahasa & Sastra',     'Sastra Arab':'Bahasa & Sastra',
    'Sastra Mandarin':'Bahasa & Sastra',   'Linguistik':'Bahasa & Sastra',
    'Ilmu Perpustakaan':'Bahasa & Sastra',
 
    'Agroteknologi':'Pertanian & Lingkungan',  'Agribisnis':'Pertanian & Lingkungan',
    'Peternakan':'Pertanian & Lingkungan',     'Perikanan':'Pertanian & Lingkungan',
    'Kehutanan':'Pertanian & Lingkungan',      'Teknologi Pangan':'Pertanian & Lingkungan',
    'Ilmu Tanah':'Pertanian & Lingkungan',
}
 
RUMPUN_LIST = sorted(set(RUMPUN_MAP.values()))
print(f'{len(JURUSAN_PROFILES)} jurusan → {len(RUMPUN_LIST)} rumpun')
print('\n'.join(f'  {i+1}. {r}' for i,r in enumerate(RUMPUN_LIST)))

95 jurusan → 10 rumpun
  1. Bahasa & Sastra
  2. Ekonomi & Bisnis
  3. Hukum & Ilmu Sosial
  4. Kesehatan & Kedokteran
  5. Komunikasi & Media
  6. Pendidikan & Keguruan
  7. Pertanian & Lingkungan
  8. Sains & MIPA
  9. Seni & Desain
  10. Teknik & Informatika


In [ ]:
#feature engineering
FEATURE_NAMES = [
    'matematika','fisika','kimia','biologi',
    'bahasa_indonesia','bahasa_inggris',
    'ekonomi','sosiologi','sejarah','seni_budaya',
    'riasec_R','riasec_I','riasec_A','riasec_S','riasec_E','riasec_C',
    #derived
    'sains_avg','sosial_avg','bahasa_avg','kreatif','logika',
    'peminatan_enc','dom_riasec_enc','riasec_dom_gap',
    'sains_vs_sosial','sains_vs_bahasa',
    'bio_vs_fis','kim_vs_mtk','bing_vs_bind',
    'top_sains_idx','top_sosial_idx',
    'riasec_R_I_diff','riasec_A_norm',
]
N_FEAT = len(FEATURE_NAMES)   #33
 
 
def build_derived(row):
    mtk,fis,kim,bio = row[0],row[1],row[2],row[3]
    bind,bing       = row[4],row[5]
    eko,sos,sej     = row[6],row[7],row[8]
    seni            = row[9]
    riasec          = np.array(row[10:16], dtype=float)
    pem             = int(row[16])
    sains  = (mtk+fis+kim+bio)/4
    sosial = (eko+sos+sej)/3
    bahasa = (bind+bing)/2
    logika = 0.6*mtk + 0.4*fis
    sr     = np.sort(riasec)[::-1]
    dom_r  = int(np.argmax(riasec))
    gap    = float(sr[0]-sr[1])
    r_max  = float(sr[0]) if sr[0]>0 else 1
    return (list(row[:16]) +
            [sains, sosial, bahasa, seni, logika, pem, dom_r, gap,
             sains-sosial, sains-bahasa, bio-fis, kim-mtk, bing-bind,
             int(np.argmax([mtk,fis,kim,bio])), int(np.argmax([eko,sos,sej])),
             float(riasec[0]-riasec[1]), float(riasec[2]/r_max)])

In [5]:
#generate data dummy
N_PER = 250
 
rows, labels = [], []
for jurusan, profile in JURUSAN_PROFILES.items():
    rumpun = RUMPUN_MAP[jurusan]
    ideal_n = np.array(profile[:10], dtype=float)
    ideal_r = np.array(profile[10:16], dtype=float)
    pem     = profile[16]
    for _ in range(N_PER):
        t = np.random.choice(['unggul','rata','lemah'], p=[0.25,0.50,0.25])
        ns = {'unggul':4.0,'rata':6.0,'lemah':8.0}[t]
        boost = {'unggul':np.random.uniform(2,5),'rata':0,'lemah':np.random.uniform(-6,-2)}[t]
        nilai  = np.clip(ideal_n + np.random.normal(boost*0.3, ns, 10), 40, 100)
        riasec = np.clip(ideal_r + np.random.normal(0, 1.0, 6), 1, 10)
        raw = list(nilai) + list(riasec) + [pem]
        rows.append(build_derived(raw))
        labels.append(rumpun)
 
df = pd.DataFrame(rows, columns=FEATURE_NAMES)
df['rumpun'] = labels
print(f'Dataset: {len(df):,} baris | Distribusi rumpun:')
print(df['rumpun'].value_counts().to_string())

Dataset: 23,750 baris | Distribusi rumpun:
rumpun
Teknik & Informatika      4000
Pendidikan & Keguruan     3000
Sains & MIPA              2500
Kesehatan & Kedokteran    2500
Hukum & Ilmu Sosial       2500
Ekonomi & Bisnis          2000
Seni & Desain             2000
Komunikasi & Media        1750
Bahasa & Sastra           1750
Pertanian & Lingkungan    1750


In [6]:
#menyimpan datarfame as csv untuk diupload di database
df.to_csv('data_dummy_rekojurusan.csv', index=False)
print("Data CSV berhasil disimpan")

Data CSV berhasil disimpan


In [7]:
#preprocessing dan splitting data
X     = df[FEATURE_NAMES].values.astype(float)
y_raw = df['rumpun'].values
le     = LabelEncoder();  y  = le.fit_transform(y_raw)
scaler = StandardScaler(); Xs = scaler.fit_transform(X)
 
X_train,X_test,y_train,y_test = train_test_split(
    Xs, y, test_size=0.25, random_state=42, stratify=y)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

Train: 17,812 | Test: 5,938


In [8]:
#training model XGBoost
print('[1/3] XGBoost ...')
xgb = XGBClassifier(
    n_estimators=500, max_depth=7, learning_rate=0.06,
    subsample=0.85, colsample_bytree=0.80,
    min_child_weight=3, gamma=0.15,
    reg_alpha=0.3, reg_lambda=1.5,
    eval_metric='mlogloss', tree_method='hist',
    random_state=42, n_jobs=-1, use_label_encoder=False)
xgb.fit(X_train, y_train, eval_set=[(X_test,y_test)], verbose=False)
acc_xgb = accuracy_score(y_test, xgb.predict(X_test))
print(f'   XGBoost   : {acc_xgb*100:.2f}%')

[1/3] XGBoost ...
   XGBoost   : 92.76%


In [9]:
#training model LightGBM
print('[2/3] LightGBM ...')
lgbm = LGBMClassifier(
    n_estimators=500, max_depth=9, num_leaves=63,
    learning_rate=0.06, subsample=0.85, colsample_bytree=0.80,
    min_child_samples=10, reg_alpha=0.3, reg_lambda=1.5,
    random_state=42, n_jobs=-1, verbose=-1)
lgbm.fit(X_train, y_train)
acc_lgbm = accuracy_score(y_test, lgbm.predict(X_test))
print(f'   LightGBM  : {acc_lgbm*100:.2f}%')

[2/3] LightGBM ...
   LightGBM  : 92.71%


In [10]:
#training model random forest
print('[3/3] Random Forest ...')
rf = RandomForestClassifier(
    n_estimators=400, max_depth=16,
    min_samples_split=4, min_samples_leaf=2,
    max_features='sqrt', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test))
print(f'   RF        : {acc_rf*100:.2f}%')

[3/3] Random Forest ...
   RF        : 91.46%


In [14]:
#model ensemble
tot = acc_xgb + acc_lgbm + acc_rf
w   = {'xgb': acc_xgb/tot, 'lgbm': acc_lgbm/tot, 'rf': acc_rf/tot}
 
p_ens = (xgb.predict_proba(X_test)*w['xgb']  +
         lgbm.predict_proba(X_test)*w['lgbm'] +
         rf.predict_proba(X_test)*w['rf'])
y_pred = np.argmax(p_ens, axis=1)
 
acc_ens  = accuracy_score(y_test, y_pred)
top3_acc = top_k_accuracy_score(y_test, p_ens, k=3)
 
print(f'{"="*50}')
print(f'  Ensemble Top-1: {acc_ens*100:.2f}%')
print(f'  Ensemble Top-3: {top3_acc*100:.2f}%')
print(f'  Bobot: XGB={w["xgb"]:.3f} | LGBM={w["lgbm"]:.3f} | RF={w["rf"]:.3f}')
print(f'{"="*50}\n')
print(classification_report(y_test, y_pred, target_names=le.classes_))

  Ensemble Top-1: 92.76%
  Ensemble Top-3: 99.76%
  Bobot: XGB=0.335 | LGBM=0.335 | RF=0.330

                        precision    recall  f1-score   support

       Bahasa & Sastra       0.92      0.92      0.92       437
      Ekonomi & Bisnis       0.98      0.98      0.98       500
   Hukum & Ilmu Sosial       0.95      0.97      0.96       625
Kesehatan & Kedokteran       0.87      0.93      0.90       625
    Komunikasi & Media       0.97      0.96      0.96       438
 Pendidikan & Keguruan       0.91      0.80      0.85       750
Pertanian & Lingkungan       0.89      0.94      0.91       438
          Sains & MIPA       0.92      0.90      0.91       625
         Seni & Desain       0.92      0.98      0.95       500
  Teknik & Informatika       0.96      0.94      0.95      1000

              accuracy                           0.93      5938
             macro avg       0.93      0.93      0.93      5938
          weighted avg       0.93      0.93      0.93      5938

